In [ ]:
#########   PHASE 1 #############
#####STEP2- Climate analysis

##Purpose
#This notebook evaluates long-term changes in annual precipitation and 
#mean surface air temperature for Lagos and Durban. 
#The analysis investigates whether climate variables exhibit significant temporal trends that may contribute to changing flood risk.

##Research Context
#Flood occurrence is often associated with climatic variability. 
#This analysis examines whether observed flood increases correspond to measurable changes in rainfall and temperature over time.

## Input Data
#precipitation data
# Mean surface air temperature data

## Methods
#Descriptive statistics
#Time series visualisation
#Mann-Kendall trend test
#Sen's slope estimation
#Linear trend analysis

## Outputs
#Climate trend plots
#Mann-Kendall statistics
#Sen's slope estimates
#Trend interpretation




#Import libraries
import pandas as pd #handles tables
import numpy as np #Like base R matrix math
import matplotlib.pyplot as plt #Base plotting engine
import seaborn as sns #prettier charts on top of matplotlib
import os #For working with file paths and folders

#Load climate data
#pd.read_excel() = readxl::read_excel() in R
#Base folder - change this to match where my  project lives
BASE_DIR = r"C:\Users\drsih\Desktop\profile\flooding_project"

#Sub-folders (raw data)
CLIMATE_DIR   = os.path.join(BASE_DIR, "data", "raw", "climate")


#Base folder - change this to match where my  project lives
BASE_DIR = r"C:\Users\drsih\Desktop\profile\flooding_project"

#Sub-folders (raw data)
CLIMATE_DIR   = os.path.join(BASE_DIR, "data", "raw", "climate")

#Durban precipitation (replace filename with your actual file)
precip_durban = pd.read_csv(
    os.path.join(CLIMATE_DIR,
                 "precipitation_annual_durban_1950_2024.csv"),
     sep=";",
    decimal=","
)

#Lagos precipitation
precip_lagos = pd.read_csv(
    os.path.join(CLIMATE_DIR, "precipitation_annual_lagos_1950_2024.csv"),
     sep=";",
    decimal=","
)

#Durban temperature
temp_durban = pd.read_csv(
    os.path.join(CLIMATE_DIR, "temperature_annual_mean_durban_1950_2024.csv"),
     sep=";",
    decimal=","
)

#Lagos temperature
temp_lagos = pd.read_csv(
    os.path.join(CLIMATE_DIR, "temperature_annual_mean_lagos_1950_2024.csv"),
     sep=";",
    decimal=","

)

print("Climate files loaded!")

In [ ]:
#Quick data check
#In R: head(), str(), summary()
#In Python: .head(), .info(), .describe()

print("\n--- Durban Precipitation: First 5 rows ---")
print(precip_durban.head())          #Shows first 5 rows (like head() in R)

print("\n--- Column names and data types ---")
print(precip_durban.info())          #Like str() in R

print("\n--- Summary statistics ---")
print(precip_durban.describe())      #Like summary() in R
print(precip_durban.describe())      #Like summary() in R
print(precip_durban.describe())      #Like summary() in R


In [ ]:
#Verification checks 
#Maki theng sure data loaded correctly before doing any analysis

datasets = {
    "Durban Precipitation" : precip_durban,
    "Lagos Precipitation"  : precip_lagos,
    "Durban Temperature"   : temp_durban,
    "Lagos Temperature"    : temp_lagos,
}

print("\n=== DATA LOADING SUMMARY ===")
for name, df in datasets.items():
    print(f"{name}:")
    print(f"  Rows: {len(df)}")               #Number of years
    print(f"  Columns: {list(df.columns)}")   #Column names
    print(f"  Year range: {df.iloc[:, 0].min()} – {df.iloc[:, 0].max()}")
    print(f"  Missing values: {df.isnull().sum().sum()}")  #Should be 0
    print()

In [ ]:
file_path = os.path.join(
    CLIMATE_DIR,
    "precipitation_annual_durban_1950_2024.csv"
)

with open(file_path, "r", encoding="utf-8") as f:
    for i in range(10):
        print(f.readline())

In [ ]:
precip_durban.head()
precip_lagos.head()
temp_durban.head()
temp_lagos.head()

In [ ]:
#Standardise the Durban precipitation dataset
#Rename columns
precip_durban = precip_durban.iloc[:, :2]
precip_durban.columns = ["Year", "Precipitation_mm"]
#Ensure correct data types
precip_durban["Year"] = pd.to_numeric(precip_durban["Year"])
precip_durban["Precipitation_mm"] = pd.to_numeric(
    precip_durban["Precipitation_mm"]
)

#Add city label
precip_durban["City"] = "Durban"

precip_durban.head()

In [ ]:
#Standardise the Lagos precipitation dataset
precip_lagos = precip_lagos.iloc[:, :2]
precip_lagos.columns = ["Year", "Precipitation_mm"]

precip_lagos["Year"] = pd.to_numeric(
    precip_lagos["Year"]
)

precip_lagos["Precipitation_mm"] = pd.to_numeric(
    precip_lagos["Precipitation_mm"]
)

precip_lagos["City"] = "Lagos"

precip_lagos.head()

In [ ]:
#Standardise Durban temperature
print(temp_durban.columns)

In [ ]:
temp_durban = temp_durban.iloc[:, :2]
temp_durban.columns = [
    "Year",
    "Temperature_C"
]

temp_durban["Year"] = pd.to_numeric(
    temp_durban["Year"]
)

temp_durban["Temperature_C"] = pd.to_numeric(
    temp_durban["Temperature_C"]
)

temp_durban["City"] = "Durban"

In [ ]:
#Standardise Lagos temperature
temp_lagos = temp_lagos.iloc[:, :2]
temp_lagos.columns = [
    "Year",
    "Temperature_C"
]

temp_lagos["Year"] = pd.to_numeric(
    temp_lagos["Year"]
)

temp_lagos["Temperature_C"] = pd.to_numeric(
    temp_lagos["Temperature_C"]
)

temp_lagos["City"] = "Lagos"

In [ ]:
#Check for missing values
print(precip_durban.isnull().sum())

print(precip_lagos.isnull().sum())

print(temp_durban.isnull().sum())

print(temp_lagos.isnull().sum())

In [ ]:
#Combine Durban and Lagos rainfall
rainfall = pd.concat(
    [precip_durban, precip_lagos],
    ignore_index=True
)

rainfall.head()

In [ ]:
#Combine Durban and Lagos temperature
temperature = pd.concat(
    [temp_durban, temp_lagos],
    ignore_index=True
)

temperature.head()

In [ ]:
#Check the combined datasets
print(rainfall.shape)

print(temperature.shape)

In [ ]:
#Save the cleaned data
rainfall.to_csv(
    "../data/clean/rainfall_clean.csv",
    index=False
)

temperature.to_csv(
    "../data/clean/temperature_clean.csv",
    index=False
)

In [ ]:
#Step 2c-Rainfall and Temperature trends (Durban vs Lagos)
import matplotlib.pyplot as plt

#Basic line plots
#Rainfall trends (1950–2024)
plt.figure(figsize=(10,5))

for city in rainfall["City"].unique():
    city_data = rainfall[rainfall["City"] == city]
    plt.plot(city_data["Year"], city_data["Precipitation_mm"], label=city)

plt.title("Rainfall Trends (1950–2024)")
plt.xlabel("Year")
plt.ylabel("Precipitation (mm)")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
#Basic line plots
#Temperature trends (1950–2024)
plt.figure(figsize=(10,5))

for city in temperature["City"].unique():
    city_data = temperature[temperature["City"] == city]
    plt.plot(city_data["Year"], city_data["Temperature_C"], label=city)

plt.title("Temperature Trends (1950–2024)")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
#STEP 2d- add trend lines to rainfall and temperature plots 
import numpy as np  # numpy gives us the polyfit() function for regression
import matplotlib.pyplot as plt
import os

#Helper function to add a trend line
#In R I'd use geom_smooth(method="lm") inside ggplot
#In Python we write a small reusable function instead

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


#Helper function: add trend line

def add_trendline(ax, x, y, color, label):
    x = np.array(x)
    y = np.array(y)

    #Fit linear model
    coeffs = np.polyfit(x, y, deg=1)
    trend_fn = np.poly1d(coeffs)

    #Sort x for clean line plotting
    x_sorted = np.sort(x)

    ax.plot(
        x_sorted,
        trend_fn(x_sorted),
        linestyle="--",
        linewidth=2,
        color=color,
        label=label
    )

    print(f"{label}: slope = {coeffs[0]:.4f} per year")


#######Colours
colors = {
    "Durban": "#1D9E75",
    "Lagos": "#378ADD"
}


##########RAINFALL

fig, ax = plt.subplots(figsize=(12, 5))

for city in rainfall["City"].unique():
    city_data = rainfall[rainfall["City"] == city].sort_values("Year")

    ax.plot(
        city_data["Year"],
        city_data["Precipitation_mm"],
        label=city,
        color=colors.get(city, "black"),
        alpha=0.6,
        linewidth=1.2
    )

    add_trendline(
        ax,
        city_data["Year"],
        city_data["Precipitation_mm"],
        color=colors.get(city, "black"),
        label=f"{city} trend"
    )

ax.set_title("Annual rainfall trends with linear regression (1950–2024)")
ax.set_xlabel("Year")
ax.set_ylabel("Precipitation (mm)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "rainfall_trends_with_regression.png"), dpi=150)
plt.show()


#############TEMPERATURE

fig, ax = plt.subplots(figsize=(12, 5))

for city in temperature["City"].unique():
    city_data = temperature[temperature["City"] == city].sort_values("Year")

    ax.plot(
        city_data["Year"],
        city_data["Temperature_C"],
        label=city,
        color=colors.get(city, "black"),
        alpha=0.6,
        linewidth=1.2
    )

    add_trendline(
        ax,
        city_data["Year"],
        city_data["Temperature_C"],
        color=colors.get(city, "black"),
        label=f"{city} trend"
    )

ax.set_title("Annual temperature trends with linear regression (1950–2024)")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature (°C)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "temperature_trends_with_regression.png"), dpi=150)
plt.show()

In [ ]:
import pandas as pd
import pymannkendall as mk #Non-parametric trend test - standard in hydrology

results = [] ##empty list to collect results


##Rainfall + Temperature loop

for variable, df, col in [
    ("Precipitation_mm", rainfall, "Precipitation_mm"),
    ("Temperature_C", temperature, "Temperature_C"),
]:

    for city in df["City"].unique():
          #Filter to one city at a time
        city_data = df[df["City"] == city].sort_values("Year")[col].values

        ##Mann-Kendall test
        result = mk.original_test(city_data)
        #result is an object with several attributes:
        #.trend  -> "increasing", "decreasing", or "no trend"
        #.p      -> p-value (like in R's MannKendall output)
        #.Tau    -> Kendall's tau correlation coefficient
        #.slope  -> Sen's slope (robust estimate of change per time step)


        results.append({
            "City": city,
            "Variable": variable,
            "Trend": result.trend, #direction
            "p-value": round(result.p, 4), #statistical signficance
            "Tau": round(result.Tau, 3),
            "Sen slope": round(result.slope, 4)  #change per year
        })

##Convert results to DataFrame---like do.call(rbind, list) in R
mk_results = pd.DataFrame(results)

print("\n=== MANN-KENDALL TREND TEST RESULTS ===")
print(mk_results.to_string(index=False))
##to_string() prints the full table without truncation

In [ ]:
##Step 2f (Decade Summary)
import os
import pandas as pd

#Ensure output folder exists
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)



rainfall = rainfall.sort_values(["City", "Year"])
temperature = temperature.sort_values(["City", "Year"])


#Rainfall: mean per decade per city
#groupby() = group_by() in R
#agg()     = summarise() in R
#round()   = round() in R

rain_decade = (
    rainfall
    .groupby(["Decade", "City"])["Precipitation_mm"]
    .agg(
        Mean_mm="mean",
        Std_mm=lambda x: x.std()
    )
    .round(1)
    .reset_index() # like ungroup() in R
)

#Pivot to wide format: cities become columns, decades become rows
#This is like tidyr::pivot_wider() in R
rain_wide = rain_decade.pivot(index="Decade", columns="City", values="Mean_mm")
rain_wide.columns.name = None   #remove the "City" label from column header
rain_wide = rain_wide.reset_index()

print("\n=== MEAN ANNUAL RAINFALL BY DECADE (mm) ===")
print(rain_wide.to_string(index=False))



#Temperature: decade summary- same structure ans rainfall

temp_decade = (
    temperature
    .groupby(["Decade", "City"])["Temperature_C"]
    .agg(Mean_C="mean")
    .round(2)
    .reset_index()
)

temp_wide = temp_decade.pivot(index="Decade", columns="City", values="Mean_C")
temp_wide.columns.name = None
temp_wide = temp_wide.reset_index()

print("\n=== MEAN ANNUAL TEMPERATURE BY DECADE (°C) ===")
print(temp_wide.to_string(index=False))



##Save to Excel
file_path = os.path.join(OUTPUT_DIR, "climate_decade_summary.xlsx")

with pd.ExcelWriter(file_path) as writer:
    rain_wide.to_excel(writer, sheet_name="Rainfall", index=False)
    temp_wide.to_excel(writer, sheet_name="Temperature", index=False)

print(f"\nSaved: {file_path}")

In [ ]:
rainfall.to_csv("outputs/rainfall_clean.csv", index=False)
temperature.to_csv("outputs/temperature_clean.csv", index=False)

In [ ]:
####Summary
##Main Outcomes
#Imported all climate datasets successfully.
#Cleaned inconsistent records and variable names.
#Generated analysis-ready datasets for climate analyses.
#Quantified long-term rainfall trends.
#Quantified long-term temperature trends.
#Evaluated statistical significance using the Mann-Kendall test.
#Estimated rates of climatic change using Sen's slope.

##Limitations
#Annual averages may mask short-duration extreme rainfall events that often trigger urban flooding.

##Next Notebook
# ->**03_urbanisation_analysis.ipynb**
#The following notebook examines urban growth and demographic change to assess how expanding urban environments influence flood vulnerability.